# Adım 1: Güvenilir model karşılaştırması

**Amaç:** Dönem başında bilinen 16 değişkenle birinci dönemde en az bir dersten kalma riskini tahmin eden modelleri karşılaştırmak.

Bu defterde algoritmaları yalnızca **eğitim verisinin 5 katlı çapraz doğrulama sonuçlarına** göre karşılaştırıyoruz. Test kümesi ayrılıyor, fakat **kullanılmıyor**.

**Uyarı:** Önceki çalışmada aynı test kümesinin sonuçlarına bakıldığı için o küme artık bütünüyle bağımsız bir nihai değerlendirme değildir. Bu defter yalnızca geliştirme aşamasında kullanılmalıdır.

Çalıştırmak için projenin kök dizinindeki data_3.csv dosyasına erişim olmalı. Bu defteri proje kökünden Jupyter ile açarsan dosya yolunu değiştirmene gerek yoktur. Alt klasörden açarsan aşağıdaki DATA_PATH değerini ../data_3.csv yap.


In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier

DATA_PATH = Path("../data_3.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("data_3.csv")
df = pd.read_csv(DATA_PATH, sep=";", encoding="utf-8-sig")
df.columns = df.columns.str.strip()
print("Ham veri:", df.shape)


## 1. Hedef ve dönem başı özellikleri

Hedef sınıfı: **1 = dönemde aldığı derslerin en az birinden kalmış**, **0 = hepsini geçmiş**.

Dönem sonu ders notu, başarı veya diğer dönem içi sonuçları girdi olarak kullanılmaz. Ders almamış öğrenciler çıkarılır.


In [ ]:
df = df.loc[df["Curricular units 1st sem (enrolled)"] > 0].copy()
y = (
    df["Curricular units 1st sem (enrolled)"]
    > df["Curricular units 1st sem (approved)"]
).astype(int)

FEATURES = [
    "Marital status", "Application mode", "Application order", "Course",
    "Daytime/evening attendance", "Previous qualification",
    "Previous qualification (grade)", "Mother's qualification",
    "Father's qualification", "Mother's occupation", "Father's occupation",
    "Admission grade", "Displaced", "Gender", "Scholarship holder",
    "Age at enrollment",
]
CATEGORICAL = [
    "Marital status", "Application mode", "Course", "Previous qualification",
    "Mother's qualification", "Father's qualification",
    "Mother's occupation", "Father's occupation",
]
NUMERIC = [c for c in FEATURES if c not in CATEGORICAL]
assert len(FEATURES) == 16
assert all(c in df.columns for c in FEATURES), "Bazı özellik sütunları eksik"
X = df[FEATURES].copy()
assert not X.isna().any().any(), "Eksik veri varsa ön işlemeye SimpleImputer ekleyin."
print("Analize giren öğrenci:", len(X))
print("Başarısız öğrenci:", int(y.sum()))
print("Başarısızlık oranı:", round(100*y.mean(), 1), "%")


## 2. Eğitim/test bölmesi

Stratify, iki grupta da başarısızlık oranını korur. **Aşağıda ayrılan test kümesi değerlendirmeye girmiyor.**


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
print("Eğitim:", X_train.shape, " / Daha sonra ele alınacak test:", X_test.shape)


## 3. Pipeline ile beş katlı çapraz doğrulama

Ön işleme işlemleri Pipeline içinde kurulur. Böylece her doğrulama katında ölçekleyici ve kategori kodlama işlemi o katın eğitim kısmından öğrenilir.


In [ ]:
def model_borusu(algoritma):
    on_isleme = ColumnTransformer([
        ("kategori", OneHotEncoder(handle_unknown="ignore", min_frequency=20), CATEGORICAL),
        ("sayi", StandardScaler(), NUMERIC),
    ])
    return Pipeline([("on", on_isleme), ("model", algoritma)])

modeller = {
    "Referans (çoğunluk)": DummyClassifier(strategy="most_frequent"),
    "Lojistik regresyon": LogisticRegression(max_iter=1500, random_state=42),
    "Karar ağacı": DecisionTreeClassifier(max_depth=5, random_state=42),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, min_samples_leaf=3, random_state=42, n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    "auc": "roc_auc", "recall": "recall",
    "precision": "precision", "f1": "f1", "accuracy": "accuracy"
}
satirlar = []
for isim, algoritma in modeller.items():
    skorlar = cross_validate(
        model_borusu(algoritma), X_train, y_train,
        cv=cv, scoring=scoring, n_jobs=1, error_score="raise"
    )
    satirlar.append({
        "Model": isim,
        "CV_AUC": skorlar["test_auc"].mean(),
        "AUC_std": skorlar["test_auc"].std(),
        "Recall": skorlar["test_recall"].mean(),
        "Precision": skorlar["test_precision"].mean(),
        "F1": skorlar["test_f1"].mean(),
        "Accuracy": skorlar["test_accuracy"].mean(),
    })

sonuclar = pd.DataFrame(satirlar).sort_values("CV_AUC", ascending=False)
print(sonuclar.round(3).to_string(index=False))


## Sonraki adım

Çıkan karşılaştırma tablosunu birlikte inceleyeceğiz.

- **CV_AUC:** İki sınıfı ayırma gücü
- **Recall:** Başarısız öğrencileri yakalama oranı (varsayılan %50 sınıflandırma eşiğinde)
- **Precision:** Riskli denilenler arasında gerçekten başarısız olanların oranı
- **F1:** Recall/precision dengesi
- **AUC_std:** Çapraz doğrulama katları arasındaki oynaklık

Model seçiminin ardından kalibrasyon, karar eşiği ve bağımsız değerlendirme planlanacak. Bu defter mevcut model.joblib ve çalışan uygulamayı değiştirmez.
